In [2]:
!pip install tldextract


   -------------------- ------------------- 1/2 [tldextract]
   -------------------- ------------------- 1/2 [tldextract]
   ---------------------------------------- 2/2 [tldextract]




[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: C:\Users\felip\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [32]:
import tldextract
import dns.resolver, dns.exception

resolver = dns.resolver.Resolver(configure=False)  # False = ignora o DNS configurado no sistema
resolver.nameservers = ["1.1.1.1", "8.8.8.8"]      # com configure=False, você precisa informar os servidores
resolver.lifetime = 5                               # tempo máximo total por consulta (segundos)

# em _query, troque dns.resolver.resolve(...) por resolver.resolve(...)

def _query(domain, rtype):
    """Retorna lista de strings, ou um status ('nxdomain', 'no_<tipo>', 'timeout', 'error: ...')."""
    try:
        resp = resolver.resolve(domain, rtype)
        if rtype == "NS":
            return [str(rr.target).rstrip(".").lower() for rr in resp]
        return [rr.to_text() for rr in resp]
    except dns.resolver.NXDOMAIN:  return "nxdomain" # domínio não existe
    except dns.resolver.NoAnswer:  return f"no_{rtype.lower()}" # existe, mas sem registro no apex
    except dns.exception.Timeout:  return "timeout" # inconclusivo
    except Exception as e:         return f"error: {e}"

def resolve_dns(domain):
    apex = tldextract.extract(domain).top_domain_under_public_suffix
    base = {"domain": domain, "apex": apex, "NS": None, "NS_apex": None, "NS_A": None, "NS_AAAA": None}

    if not apex:
        return {**base, "NS": "invalid"}  # mesmas chaves em todos os retornos

    NS = _query(apex, "NS")
    if not isinstance(NS, list):          # NS é um status de erro: não há hostnames para consultar
        return {**base, "NS": NS}

    return {
        **base,                            # {**a, k: v} copia o dict e sobrescreve só as chaves dadas
        "NS": NS,
        "NS_apex": [tldextract.extract(ns).top_domain_under_public_suffix for ns in NS],
        "NS_A":    [_query(ns, "A") for ns in NS],
        "NS_AAAA": [_query(ns, "AAAA") for ns in NS],
    }

teste = "www.banrisul.com.br"
print(resolve_dns(teste))

{'domain': 'www.banrisul.com.br', 'apex': 'banrisul.com.br', 'NS': ['ns0.banrisul.com.br', 'ns1.banrisul.com.br'], 'NS_apex': ['banrisul.com.br', 'banrisul.com.br'], 'NS_A': [['201.55.240.11'], ['201.55.240.12']], 'NS_AAAA': [['2801:80:280:f940::11'], ['2801:80:280:c016::12']]}
